# Homework 7: Linear Regression and Model Evaluation

This homework covers three lectures:
* **Lecture 19** (linear regression) -- Problems 1-2
* **Lecture 20** (model evaluation, train/test splits, overfitting) -- Problem 4
* **Both** -- Problem 3 asks you to use Lecture 19's tools to make a Lecture-20-flavored
  judgment call

Two datasets:
* `dataset_hall_petch` -- 32 specimens, the canonical grain-size/yield-strength relationship
  (Lecture 19's opening hook)
* `dataset_steels` -- 915 steel alloys, composition and mechanical properties (familiar since
  Lecture 15, reused all through Lectures 19-21)

## Submission instructions

Upload the `ipynb` file to Canvas:
> File -> Download -> ipynb -> upload to Canvas (like any other file)

*Only* the `ipynb` file type will be accepted.

Save a copy of the notebook right away to avoid losing your work!

# Problem 0 (0 pts): generative AI usage statement

As you work on this assignment, feel free to use generative AI tools to help you learn,
understand, and debug Python code. In particular, you could get hints or conceptual guidance in
the implementation you write yourself.

However, you must clearly disclose and cite all use of AI. You must include:
1. The name(s) of the AI tool(s) used.
2. The specific prompt(s) you used to generate the content.
3. A description of how you used the output and what edits or additions you made to integrate it
   into your own work.

You are fully responsible for the final submitted work -- critically evaluate, fact-check, and
verify all AI-generated content for validity. Failure to properly cite and disclose AI use
constitutes plagiarism under Penn State's Academic Integrity policy.

Write your disclosure (or "I did not use an AI tool for this assignment") in the cell below.

*your disclosure here*

## Data file for Problem 1

## Dataset: Hall-Petch Grain Size vs. Yield Strength (synthetic, calibrated)

The Hall-Petch relationship, sigma_y = sigma_0 + k * d^(-1/2), is one of the most
famous single lines in materials science: smaller grains (more grain-boundary
area per unit volume, more obstacles to dislocation motion) give a higher yield
strength. Because it's linear in d^(-1/2), fitting it is a one-line `linregress`
call if the x-axis is pre-transformed -- no curve-fitting machinery required.

**Why synthetic, not sourced.** A candidate real dataset (sp8rks/MaterialsInformatics,
MIT-licensed) was considered and rejected: its underlying xlsx-derived numbers have
*unverified* provenance (no traceable original measurement source). Rather than ship
numbers we can't stand behind, this dataset is synthesized from **published Hall-Petch
constants for annealed/pure copper**: sigma_0 (≈25 MPa) and k (≈0.11 MPa*m^0.5)
— see e.g. Hansen, "Hall-Petch relation and boundary strengthening," Scripta Materialia
51 (2004) 801-806, and standard references on annealed Cu polycrystals.

**Synthesis parameters (reproducible, seed=219):**
- 32 specimens, grain size drawn uniformly from 5-100 micrometres
- `inv_sqrt_grain_size` = 1 / sqrt(grain_size_um * 1e-6) in **m^-0.5** (grain size
  converted to metres before the inverse-sqrt transform, so the fitted slope comes
  out directly in the textbook units MPa*m^0.5 -- matching how k is normally quoted)
- clean signal: yield_strength_mpa = 25 + 0.11 * inv_sqrt_grain_size
- Gaussian noise, std = 1.8 MPa, added to yield strength
- **Verified:** `scipy.stats.linregress(inv_sqrt_grain_size, yield_strength_mpa)`
  recovers slope ≈0.111 MPa*m^0.5 (target 0.11), intercept ≈25.2 MPa (target 25),
  R^2 ≈0.94 (target range 0.90-0.97) -- see
  `docs/2026-07-22-new-dataset-verification.md` for the full run.

In [1]:
import os
import pandas as pd

_file = 'hall_petch.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}', f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    data = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e

x = data['inv_sqrt_grain_size']
y = data['yield_strength_mpa']

print(f'{len(data)} specimens, grain size {data["grain_size_um"].min():.1f}-{data["grain_size_um"].max():.1f} um')
data.head()

32 specimens, grain size 6.6-98.8 um


,grain_size_um,inv_sqrt_grain_size,yield_strength_mpa
0,6.6,389.249,69.71
1,7.3,370.117,62.43
2,8.9,335.201,64.55
3,9.7,321.081,60.68
4,10.8,304.290,54.79


# Problem 1 (25 pts): the Hall-Petch anchor

The Hall-Petch relation says yield strength is linear in `1/sqrt(grain size)`, with slope `k`
and intercept `sigma_0` -- exactly the shape fit in Lecture 19's opener.

(a) Fit `yield_strength_mpa ~ inv_sqrt_grain_size` with `scipy.stats.linregress`. Print the
slope, intercept, and R².

(b) In a sentence each: how does your fitted slope compare to the published k ≈ 0.11
MPa·√m for annealed copper, and your fitted intercept to the published sigma_0 ≈ 25 MPa? Is
this the "tight" or the "modest" kind of fit, and how do you know from the R² alone?

(c) The dataset ships `inv_sqrt_grain_size` pre-computed, but a real specimen report usually
just gives you a grain size in micrometres. For a new specimen with grain size = 15 µm,
compute `inv_sqrt_grain_size` yourself (convert micrometres to metres first, then take
`1/sqrt(...)`), then use your fitted line to predict its yield strength in MPa.

## Data file for Problems 2-4

Run this code cell to load the same steel alloy dataset from Lectures 15/19-21 -- 915 alloys,
composition in weight percent, mechanical properties, and an `Alloy family` column. **Do not
drop any rows here** -- Problem 3 specifically needs the data exactly as loaded.

## Dataset: Steel Alloy Compositions and Mechanical Properties

This dataset contains elemental compositions and mechanical properties of 915 steel alloys.

Composition columns (in wt%): `C`, `Si`, `Mn`, `P`, `S`, `Ni`, `Cr`, `Mo`, `Cu`, `V`, `Al`, `N`, `Ceq`, `Nb + Ta`

Target columns: `0.2% Proof Stress (MPa)`, `Tensile Strength (MPa)`, `Elongation (%)`, `Reduction in Area (%)`

The `Alloy code` column contains labels like `A1`, `B3`, etc. We derive `alloy_family` (first letter only)
to get a small set of categorical labels useful for classification tasks (4 families: C, L, M, V, all >150 samples).

In [2]:
import os
import pandas as pd

_file = 'steels.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}', f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    data = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e

# derive alloy family label (first letter of alloy code)
data['Alloy family'] = [c[0] for c in data['Alloy code']]

# set up features (composition columns) and target
x = data.loc[:, ' C':'Nb + Ta']
y = data['Alloy family']
alloy_family = data['Alloy family']

print(f'{len(data)} samples, {x.shape[1]} composition features')
data.head()

915 samples, 14 composition features


,Alloy code,C,Si,Mn,P,S,Ni,Cr,Mo,Cu,...,Al,N,Ceq,Nb + Ta,Temperature (°C),0.2% Proof Stress (MPa),Tensile Strength (MPa),Elongation (%),Reduction in Area (%),Alloy family
0,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,27,342,490,30,71,M
1,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,100,338,454,27,72,M
2,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,200,337,465,23,69,M
3,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,300,346,495,21,70,M
4,MBB,0.12,0.36,0.52,0.009,0.003,0.089,0.97,0.61,0.04,...,0.003,0.0066,0.0,0.0,400,316,489,26,79,M


# Problem 2 (25 pts): a fresh property pair

Neither lecture fit this pair -- you're transferring the skill, not repeating a demo.

(a) Fit `' Elongation (%)'` (x, predictor) against `' Reduction in Area (%)'` (y, response)
with `scipy.stats.linregress`. Print the slope, intercept, and R².

(b) In one sentence: both columns measure ductility (how much an alloy deforms before
fracture). Does the *sign* of your slope make physical sense for two measures of the same
underlying property? Why or why not?

(c) Predict `' Reduction in Area (%)'` for an alloy with 25% elongation, using your fitted
line.

# Problem 3 (25 pts): is this fit honest?

This dataset has a known data-entry problem, and you're going to find it yourself instead of
being told where it is.

(a) Fit `' 0.2% Proof Stress (MPa)'` (x) against `' Tensile Strength (MPa)'` (y) with
`scipy.stats.linregress`, using `data` exactly as loaded (do not drop anything yet). Print the
slope, intercept, and R².

(b) Something in this fit should look off. Use
`data.sort_values(' Tensile Strength (MPa)', ascending=False).head()` to look at the highest
tensile-strength rows. Identify the one row that looks like a data-entry error rather than a
real measurement, and explain in one sentence what makes it implausible (compare it to the
next-highest value in the column).

(c) Drop that one row (`data = data.drop(index=...)`), re-fit the exact same pair, and print
the new slope, intercept, and R².

(d) In 2-3 sentences: your R² changed substantially between (a) and (c) because of a single
row out of 915. If someone asked you "how well does proof stress predict tensile strength for
these alloys," which R² would you report, and why? Is reporting the number from (a) without
mentioning the dropped row an honest thing to do?

# Problem 4 (25 pts): a polynomial-degree study

Use the `data` you cleaned in Problem 3(c) for everything below (it should still have the bad
row removed).

(a) Redefine `x` and `y` from the cleaned `data`: `x` should be all 14 composition columns
(`data.loc[:, ' C':'Nb + Ta']`), `y` should be `data[' 0.2% Proof Stress (MPa)']`. This
replaces the dataset module's default `x`/`y` -- exactly what Lecture 20 did.

(b) Run the include cell below to get `idx_train`, `idx_test`, `xtrain`, `xtest`, `ytrain`,
`ytest` -- the same split tool Lecture 20 used.

## Train/Test Split

In order to evaluate the performance of a model on unseen data, we need
to *hold out* some data. This will be called the **test** data.
The data used to fit will be called the **training** data.

As we use more sophisticated models, it is vitally important that we
evaluate our models in this way to avoid *overfitting*.
Remember that a model with enough degrees of freedom can perfectly fit
any data, but it won't have any predictive power!

```
all data:    [########################################################]
shuffle, then slice into two pieces:
train (75%): [##########################################]
test  (25%):                                              [##########]
```

We shuffle the *indices* once, then slice both `x` and `y` with that same shuffled order --
so a row's features and its target always stay paired.

In [3]:
import numpy as np

# split indices rather than data -- more flexible
rng = np.random.default_rng(0)
shuffled = rng.permutation(data.index)
n_test = int(np.ceil(0.25 * len(shuffled)))
idx_test = shuffled[:n_test]
idx_train = shuffled[n_test:]

xtrain = x.loc[idx_train]
xtest  = x.loc[idx_test]

ytrain = y.loc[idx_train]
ytest  = y.loc[idx_test]

print(f"Train: {xtrain.shape[0]} samples, Test: {xtest.shape[0]} samples")

Train: 686 samples, Test: 229 samples


(c) Build a single-feature series for **`' Ni'`** (nickel content) -- a different feature than
Lecture 20's chromium demo. Following Lecture 20's Section 5 exactly: take only the *first 20*
rows of the training set (`idx_train[:20]`) as your training data, and fit `np.polyfit(x, y,
d)` / `np.polyval` for `d in [1, 2, 3, 5, 8]`. For each degree, print the train RMSE (on your
20-point training set) and test RMSE (on the full held-out test set).

(d) Plot train RMSE and test RMSE vs. degree on one axes, with a log-scaled y-axis (`plt.yscale
('log')`) -- the same style as Lecture 20's plot.

(e) In one sentence: at which degree does test RMSE clearly break away from train RMSE, and
is that the same degree where train RMSE is lowest? Why does that gap matter more than either
number alone?

# Wrap-up

This homework asked for exactly what Lectures 19-20 built: fit a line and read its
slope/intercept/R² in physical units (Problems 1-2), notice when a fit-quality number is being
distorted by bad data rather than genuine scatter (Problem 3), and diagnose overfitting from a
train/test error curve you built yourself (Problem 4). If your R² changed a lot when you
dropped one row in Problem 3, that's not a bug in the assignment -- that's the whole point:
a single number never tells you whether to trust it. Always look at the data behind it.